# Pipeline A — Step 1: Annual Panel Aggregation

**Filosofi**: Data tahunan (x3-x10) harus dimodelkan secara tahunan. Tidak membuat ilusi variasi bulanan.

**Transformasi:**
- Target `twp90_pct` → rata-rata tahunan per provinsi
- Fitur bulanan (BI rate, inflasi) → rata-rata tahunan per provinsi
- Fitur tahunan → diambil langsung (first/unique per tahun)
- `x4_tpt_pct` → rata-rata Feb + Agt per provinsi per tahun
- Log-transform pada PDRB dan tabungan

**Input:** `/output/1_raw_panel_data.csv`

**Output:** `pipeline_A/output/A1_annual_panel.csv`

In [58]:
from pathlib import Path
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start, *start.parents]:
        if (p / 'pipeline_A' / 'output' / 'A0_raw_panel_data.csv').exists():
            return p
    raise FileNotFoundError('Could not find A0_raw_panel_data.csv')

ROOT = find_project_root(Path.cwd())
input_path = ROOT / 'pipeline_A' / 'output' / 'A0_raw_panel_data.csv'
output_dir = ROOT / 'pipeline_A' / 'output'
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / 'A1_annual_panel.csv'

df = pd.read_csv(input_path)
df['tanggal'] = pd.to_datetime(df['tanggal'])
print(f'Loaded: {input_path}')
print(f'Shape: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Years: {sorted(df["tahun"].unique())}')
print(f'Provinces: {df["provinsi_id"].nunique()}')

Loaded: C:\Users\bimyu\Documents\Projects\DatathonMETC\pipeline_A\output\A0_raw_panel_data.csv
Shape: 1,488 rows x 16 cols
Years: [np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]
Provinces: 31


In [59]:
# === Agregasi ke level tahunan ===
df_annual = df.groupby(['provinsi_id', 'nama_provinsi', 'tahun']).agg(
    twp90_avg       = ('twp90_pct', 'mean'),
    twp90_std       = ('twp90_pct', 'std'),
    twp90_max       = ('twp90_pct', 'max'),
    twp90_min       = ('twp90_pct', 'min'),
    x1_bi_rate_avg  = ('x1_bi_rate_pct', 'mean'),
    x2_inflasi_avg  = ('x2_inflasi_yoy', 'mean'),
    x2_inflasi_std  = ('x2_inflasi_yoy', 'std'),
    x3_pdrb         = ('x3_pdrb_per_kapita', 'first'),
    x4_tpt          = ('x4_tpt_pct', 'mean'),          # mean of Feb+Aug values
    x5_internet     = ('x5_penetrasi_internet_pct', 'first'),
    x6_tabungan     = ('x6_tabungan_miliar', 'first'),
    x7_kc_bank      = ('x7_jumlah_kc_bank', 'first'),
    x8_ldr          = ('x8_ldr_pct', 'first'),
    x9_npl          = ('x9_npl_ratio', 'first'),
    x10_umkm        = ('x10_rasio_umkm', 'first'),
).reset_index()

print(f'Annual panel shape: {df_annual.shape[0]} rows x {df_annual.shape[1]} cols')
print(f'Expected: {df["provinsi_id"].nunique()} provinces x {df["tahun"].nunique()} years = {df["provinsi_id"].nunique() * df["tahun"].nunique()}')
display(df_annual.head())

Annual panel shape: 124 rows x 18 cols
Expected: 31 provinces x 4 years = 124


,provinsi_id,nama_provinsi,tahun,twp90_avg,twp90_std,twp90_max,twp90_min,x1_bi_rate_avg,x2_inflasi_avg,x2_inflasi_std,x3_pdrb,x4_tpt,x5_internet,x6_tabungan,x7_kc_bank,x8_ldr,x9_npl,x10_umkm
0,1,Banten,2022,0.025142,0.002400,0.0287,0.0217,4.000000,0.004792,0.004956,61414000.0,8.310,0.8100,236586.03,96.0,0.7703,0.0202,0.2979
1,1,Banten,2023,0.029208,0.012290,0.0513,0.0216,5.812500,0.002412,0.002024,66147000.0,7.745,0.8910,246873.64,94.0,0.7934,0.0173,0.2995
2,1,Banten,2024,0.022292,0.002787,0.0266,0.0189,6.104167,0.025075,0.005433,70276000.0,6.850,0.8455,268171.77,96.0,0.7943,0.0232,0.2826
3,1,Banten,2025,0.024117,0.002899,0.0274,0.0199,5.291667,0.017342,0.009333,74673000.0,6.665,0.8399,268278.77,97.0,0.7998,0.0281,0.2756
4,2,Dki Jakarta,2022,0.025492,0.004145,0.0336,0.0205,4.000000,0.003458,0.003905,299675000.0,7.590,0.8340,3393358.58,451.0,0.9222,0.0217,0.0461


In [60]:
# === Log-transform variabel dengan skala besar ===
df_annual['log_pdrb'] = np.log1p(df_annual['x3_pdrb'])
df_annual['log_tabungan'] = np.log1p(df_annual['x6_tabungan'])
df_annual['log_kc_bank'] = np.log1p(df_annual['x7_kc_bank'])

# === Lag tahunan (t-1) pada fitur kunci ===
lag_features = ['twp90_avg', 'x1_bi_rate_avg', 'x2_inflasi_avg',
                'x9_npl', 'x8_ldr', 'log_pdrb']
for feat in lag_features:
    df_annual[f'{feat}_lag1'] = df_annual.groupby('provinsi_id')[feat].shift(1)

# === Delta features (year-over-year change) ===
df_annual['delta_twp90'] = df_annual.groupby('provinsi_id')['twp90_avg'].diff()
df_annual['delta_npl'] = df_annual.groupby('provinsi_id')['x9_npl'].diff()
df_annual['delta_pdrb_pct'] = df_annual.groupby('provinsi_id')['x3_pdrb'].pct_change()

# === Missing summary ===
print('\n=== Missing Values ===')
missing = df_annual.isnull().sum()
display(missing[missing > 0])


=== Missing Values ===


twp90_avg_lag1         31
x1_bi_rate_avg_lag1    31
x2_inflasi_avg_lag1    31
x9_npl_lag1            31
x8_ldr_lag1            31
log_pdrb_lag1          31
delta_twp90            31
delta_npl              31
delta_pdrb_pct         31
dtype: int64

In [61]:
# === Descriptive Statistics ===
print('=== Descriptive Statistics ===')
display(df_annual.describe().round(4))

# === Save ===
df_annual.to_csv(output_path, index=False)
print(f'\nSaved: {output_path}')
print(f'Columns: {list(df_annual.columns)}')

=== Descriptive Statistics ===


,provinsi_id,tahun,twp90_avg,twp90_std,twp90_max,twp90_min,x1_bi_rate_avg,x2_inflasi_avg,x2_inflasi_std,x3_pdrb,...,log_kc_bank,twp90_avg_lag1,x1_bi_rate_avg_lag1,x2_inflasi_avg_lag1,x9_npl_lag1,x8_ldr_lag1,log_pdrb_lag1,delta_twp90,delta_npl,delta_pdrb_pct
count,124.0000,124.0000,124.0000,124.0000,124.0000,124.0000,124.0000,124.0000,124.0000,1.240000e+02,...,124.0000,93.0000,93.0000,93.0000,93.0000,93.0000,93.0000,93.0000,93.0000,93.0000
mean,16.4194,2023.5000,0.0208,0.0033,0.0268,0.0166,5.3021,0.0127,0.0071,8.008814e+07,...,4.3019,0.0207,5.3056,0.0103,0.0229,1.1563,17.9820,-0.0003,0.0009,0.0672
std,9.4077,1.1226,0.0087,0.0033,0.0131,0.0074,0.8094,0.0099,0.0040,6.227377e+07,...,0.8180,0.0087,0.9359,0.0100,0.0083,0.4718,0.5582,0.0060,0.0062,0.0422
min,1.0000,2022.0000,0.0070,0.0006,0.0115,0.0044,4.0000,0.0013,0.0013,2.165800e+07,...,2.8904,0.0070,4.0000,0.0013,0.0107,0.5920,16.8909,-0.0151,-0.0227,-0.0967
25%,8.0000,2022.7500,0.0149,0.0019,0.0195,0.0119,4.9688,0.0034,0.0044,4.784675e+07,...,3.8067,0.0148,4.0000,0.0026,0.0168,0.8306,17.6461,-0.0033,-0.0013,0.0573
50%,16.0000,2023.5000,0.0192,0.0026,0.0235,0.0150,5.5521,0.0084,0.0065,6.397700e+07,...,4.1271,0.0195,5.8125,0.0048,0.0208,1.0552,17.9574,-0.0006,0.0014,0.0667
75%,25.0000,2024.2500,0.0258,0.0037,0.0322,0.0204,5.8854,0.0225,0.0091,7.588350e+07,...,4.5875,0.0256,6.1042,0.0212,0.0269,1.2836,18.1138,0.0021,0.0036,0.0767
max,32.0000,2025.0000,0.0650,0.0323,0.1158,0.0580,6.1042,0.0343,0.0188,3.676870e+08,...,6.1137,0.0650,6.1042,0.0343,0.0538,2.9248,19.6573,0.0342,0.0190,0.3600



Saved: C:\Users\bimyu\Documents\Projects\DatathonMETC\pipeline_A\output\A1_annual_panel.csv
Columns: ['provinsi_id', 'nama_provinsi', 'tahun', 'twp90_avg', 'twp90_std', 'twp90_max', 'twp90_min', 'x1_bi_rate_avg', 'x2_inflasi_avg', 'x2_inflasi_std', 'x3_pdrb', 'x4_tpt', 'x5_internet', 'x6_tabungan', 'x7_kc_bank', 'x8_ldr', 'x9_npl', 'x10_umkm', 'log_pdrb', 'log_tabungan', 'log_kc_bank', 'twp90_avg_lag1', 'x1_bi_rate_avg_lag1', 'x2_inflasi_avg_lag1', 'x9_npl_lag1', 'x8_ldr_lag1', 'log_pdrb_lag1', 'delta_twp90', 'delta_npl', 'delta_pdrb_pct']
